[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svngoku/cybersup-deep-learning-intro/blob/main/notebooks/etudiants/03_transfer_learning.ipynb)


# TP 03 — Transfert source 0–4 vers cible 5–9
**180 minutes · CPU · préentraînement local réel**

Comparer trois protocoles à faible quantité de labels cible. Ce TP n'utilise pas de poids ImageNet.
Le coût source (15 époques) est distinct du budget cible (45 époques). La source et la cible n'ont aucune classe commune.
Découpage : préparation 25 min, source 35 min, stratégies 55 min, analyse 35 min, restitution 30 min.

## Préparation de l'environnement

Exécuter cette cellule en premier dans un runtime Python 3 sur **CPU**. Les bibliothèques déjà installées sont conservées ; seules celles qui manquent sont ajoutées. Aucun clonage, fichier voisin ou montage Drive n'est nécessaire. Enregistrer une copie du notebook avant de commencer. Si le lien vers le dépôt privé est inaccessible, importer le fichier fourni dans le pack étudiant.


In [ ]:
# Cellule autonome : les bibliothèques déjà présentes dans Colab sont conservées.
import importlib.util
import json
import os
import platform
import subprocess
import sys

DEPENDENCIES = {
    "numpy": "numpy==2.2.6",
    "sklearn": "scikit-learn==1.7.2",
    "torch": "torch==2.8.0",
    "matplotlib": "matplotlib==3.10.6",
}
missing = [pin for module, pin in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des bibliothèques absentes :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import matplotlib
import numpy as np
import sklearn
import torch

ENVIRONMENT = {
    "runtime": "google-colab" if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ else "python-local-ou-ci",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "device": "cpu",
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))
print("Les cinq TP utilisent le CPU, même si un GPU est disponible.")


In [ ]:
import os, copy, math, random
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')
print('Référence : CPU | PyTorch', torch.__version__, '| graine', SEED)

In [ ]:
def evaluate(model, X, y):
    model.eval()
    with torch.inference_mode():
        logits = model(X)
        loss = nn.functional.cross_entropy(logits, y).item()
        pred = logits.argmax(-1).numpy()
    return loss, accuracy_score(y.numpy(), pred), pred

def fit(model, X, y, Xv, yv, epochs=12, lr=1e-3, frozen=False):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    best_loss = float('inf'); best = copy.deepcopy(model.state_dict()); history=[]
    gen = torch.Generator().manual_seed(SEED)
    for epoch in range(epochs):
        model.train()
        if frozen: model.features.eval()
        total=0.0
        for ids in torch.randperm(len(y), generator=gen).split(64):
            opt.zero_grad(set_to_none=True)
            loss=nn.functional.cross_entropy(model(X[ids]),y[ids])
            loss.backward(); opt.step(); total += loss.item()*len(ids)
        vl,va,_=evaluate(model,Xv,yv)
        history.append((total/len(y),vl,va))
        if vl < best_loss: best_loss=vl; best=copy.deepcopy(model.state_dict())
    model.load_state_dict(best)
    return np.array(history)

class TinyCNN(nn.Module):
    def __init__(self, classes=10):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                                   nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten())
        self.head=nn.Linear(64,classes)
    def forward(self,x): return self.head(self.features(x))

In [ ]:
ds=load_digits(); X=ds.images.astype('float32')[:,None]/16; y=ds.target
source_ids=np.flatnonzero(y<5); target_ids=np.flatnonzero(y>=5)
assert not(set(source_ids)&set(target_ids))
st,sv=train_test_split(source_ids,test_size=.2,stratify=y[source_ids],random_state=SEED)
tt,te=train_test_split(target_ids,test_size=.2,stratify=y[target_ids],random_state=SEED)
tt,tv=train_test_split(tt,test_size=.25,stratify=y[tt],random_state=SEED)
rng=np.random.default_rng(SEED)
few=np.concatenate([rng.choice(tt[y[tt]==k],20,replace=False) for k in range(5,10)])
assert not(set(few)&set(tv) or set(few)&set(te) or set(tv)&set(te))
def tensors(ids,offset=0): return torch.tensor(X[ids]),torch.tensor(y[ids]-offset,dtype=torch.long)
Xs,ys=tensors(st); Xsv,ysv=tensors(sv)
Xt,yt=tensors(few,5); Xv,yv=tensors(tv,5); Xe,ye=tensors(te,5)
source=TinyCNN(classes=5)
source_history=fit(source,Xs,ys,Xsv,ysv,epochs=15)
print('Source validation :',evaluate(source,Xsv,ysv)[:2])
print('Cible :',len(yt),'train ;',len(yv),'validation ;',len(ye),'test')

In [ ]:
def target_model(pretrained):
    torch.manual_seed(SEED)
    m=TinyCNN(classes=5)
    if pretrained: m.features.load_state_dict(source.features.state_dict())
    return m

scratch=target_model(False)
hs=fit(scratch,Xt,yt,Xv,yv,epochs=45,lr=1e-3)

frozen=target_model(True)
for p in frozen.features.parameters(): p.requires_grad_(False)
hf=fit(frozen,Xt,yt,Xv,yv,epochs=45,lr=1e-3,frozen=True)

finetune=target_model(True)
for p in finetune.features.parameters(): p.requires_grad_(False)
h1=fit(finetune,Xt,yt,Xv,yv,epochs=35,lr=1e-3,frozen=True)
before=copy.deepcopy(finetune.state_dict()); before_loss=evaluate(finetune,Xv,yv)[0]
for p in finetune.features.parameters(): p.requires_grad_(True)
h2=fit(finetune,Xt,yt,Xv,yv,epochs=10,lr=3e-4)
if evaluate(finetune,Xv,yv)[0]>before_loss: finetune.load_state_dict(before)

models={'depuis_zero':scratch,'gele':frozen,'fine_tuning':finetune}
for name,m in models.items(): print(name,'validation :',evaluate(m,Xv,yv)[:2])

## Travail demandé
1. Expliquer le remappage des labels cible et le remplacement de la tête.
2. Documenter le nombre de paramètres entraînables et le coût de chaque phase.
3. Refaire la comparaison avec 10 puis 40 exemples par classe, sur la validation.
4. Proposer une source peu pertinente et discuter le risque de transfert négatif.
5. Expliquer ce qui changerait avec BatchNorm : gel des poids ≠ gel des statistiques.

Le nombre d'époques cible est aligné, pas nécessairement le coût total en temps ou en calcul arrière.
Les stratégies sont fixées avant la lecture du test ; aucune n'est choisie à partir de ce test.

In [ ]:
RESULT={'tp':3,'source_epochs':15,'target_epochs':45,'target_train_examples':len(yt),'arms':{}}
for name,m in models.items():
    loss,acc,pred=evaluate(m,Xe,ye)
    RESULT['arms'][name]={'test_loss':loss,'test_accuracy':acc,'macro_f1':f1_score(ye.numpy(),pred,average='macro')}
print(RESULT)

## Exporter les résultats

Exécuter la cellule suivante après le bilan terminal pour enregistrer métriques, graine et versions dans un fichier JSON. Dans Colab, activer `DOWNLOAD_RESULTS` pour le télécharger. Sauvegarder également une copie du notebook avec les sorties et les réponses aux investigations. Les fichiers du runtime sont temporaires.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

DOWNLOAD_RESULTS = False  # @param {type:"boolean"}

def json_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Valeur non sérialisable : {type(value).__name__}")

result_path = Path("resultats") / f"tp_{RESULT['tp']:02d}.json"
result_path.parent.mkdir(exist_ok=True)
experiment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "environment": ENVIRONMENT,
    "results": RESULT,
}
result_path.write_text(
    json.dumps(experiment, indent=2, ensure_ascii=False, default=json_scalar) + "\n",
    encoding="utf-8",
)
print("Résultats enregistrés :", result_path)

if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab, récupérer le fichier dans le dossier resultats/.")
    else:
        files.download(str(result_path))
